## Results

Find product ids for focus products.

In [7]:
%run ../utils/sampling.py
import pandas as pd

product_info_df = pd.read_csv('../data/cleaned/product-features.csv')
departments = pd.read_csv('../dataset/departments.csv')
aisles = pd.read_csv('../dataset/aisles.csv')
products = pd.read_csv('../dataset/products.csv')

full_products_df = product_info_df.merge(products, on='product_id', how='left')

product_names_to_search = [
    "Coke",
    "Bananas",
    "Organic Whole Milk",
    "Plain Bagels",
    "Spaghetti"
]

top_ids = get_top_product_ids(full_products_df, product_names_to_search)
print(top_ids)

found_products = search_products(full_products_df, "Coke")
print(found_products.head(1))

found_products = search_products(full_products_df, "Eggo")
print(found_products.head(1))

found_products = search_products(full_products_df, "potato chips")
print(found_products.head(1))

{'Coke': {'query': 'Coke', 'matched_name': 'Diet Coke', 'product_id': 43631, 'order_penetration_pct': 0.2090284098225933}, 'Bananas': {'query': 'Bananas', 'matched_name': 'Banana', 'product_id': 24852, 'order_penetration_pct': 14.69933191782944}, 'Organic Whole Milk': {'query': 'Organic Whole Milk', 'matched_name': 'Organic Whole Milk', 'product_id': 27845, 'order_penetration_pct': 4.289592686991776}, 'Plain Bagels': {'query': 'Plain Bagels', 'matched_name': 'Plain Bagels', 'product_id': 20738, 'order_penetration_pct': 0.3195770658507922}, 'Spaghetti': {'query': 'Spaghetti', 'matched_name': 'Spaghetti', 'product_id': 32734, 'order_penetration_pct': 0.49186997686379}}
   product_name  product_id  order_penetration_pct
0  Coke Classic       16696               0.335068
             product_name  product_id  order_penetration_pct
0  Eggo Homestyle Waffles       30696               0.276216
                      product_name  product_id  order_penetration_pct
0  Sea Salt & Vinegar Potato C

Build dataframe of focus products and save.

In [8]:
product_ids = [
    info["product_id"] 
    for info in top_ids.values() 
    if info is not None
]

product_ids.append(16696)
product_ids.append(30696)
product_ids.append(40709)
print(product_ids)
sampled_products_df = pd.DataFrame(product_ids, columns=['product_id'])
sampled_products_df.to_csv("../results/sampled-products.csv", index=None)

[43631, 24852, 27845, 20738, 32734, 16696, 30696, 40709]


Compute pairwise probabilities for focus products.

In [9]:
%run ../utils/pairwise.py

# Get product_ids as a series
products = product_ids.copy()
product_df = pd.read_csv('../data/cleaned/product-info-full.csv')
orders_full_df = pd.read_csv('../dataset/order_products__prior.csv')

order_product_df = orders_full_df[['order_id', 'product_id']]

# Compute pairwise probabilties for focus products
compute_pairwise_probabilities_sample(order_product_df, 
    products,
    output_csv="../results/products-pairwise.csv",
    batch_size=1000
)

100%|██████████| 1/1 [00:04<00:00,  4.05s/it]

Completed computation. Saved to ../results/products-pairwise.csv


Compute substitution score for focus products.

In [10]:
%run ../utils/substitutes.py

dept1_df = pd.read_csv('../data/validation/pairwise/pairwise-dept1.csv')
dept3_df = pd.read_csv('../data/validation/pairwise/pairwise-dept3.csv')
dept4_df = pd.read_csv('../data/validation/pairwise/pairwise-dept4.csv')
dept7_df = pd.read_csv('../data/validation/pairwise/pairwise-dept7.csv')
dept9_df = pd.read_csv('../data/validation/pairwise/pairwise-dept9.csv')
dept16_df = pd.read_csv('../data/validation/pairwise/pairwise-dept16.csv')
dept19_df = pd.read_csv('../data/validation/pairwise/pairwise-dept19.csv')

pairwise_df = pd.concat([dept1_df, dept3_df, dept4_df, dept7_df, dept9_df, dept16_df, dept19_df], ignore_index=True)

similarity_df = pd.read_csv('../data/cleaned/product-similiarity.csv')

compute_sub_score(
    product_df,
    pairwise_df,
    products,
    similarity_df,
    "../results/raw-substitutes.csv")

Completed substitute calculations. Saved to ../results/raw-substitutes.csv


Compute transferability for focus products.

In [11]:
%run ../utils/substitutes.py
%run ../utils/results.py

substitutes_df = pd.read_csv("../results/raw-substitutes.csv")

# Find the best substitution score threshold which will identify a product as a true substitute
best_threshold = 0.4

# Add a new column 'identified_substitute' based on the threshold
substitutes_df['identified_substitute'] = substitutes_df['score'] >= best_threshold

# Compute transferability %
results = compute_transferability(substitutes_df, top_n=5)
results.to_csv("../results/substitutes-transfer.csv", index=False)

# Print the results
show_sub_results(results)


Product: Coke Classic  (ID: 16696)


,sub_name,transferability_pct,score,aisle
0,Classic Soda,0.149826,0.701174,soft drinks
1,Coke Zero,0.144124,0.674490,soft drinks
2,Coke,0.141851,0.663852,soft drinks
3,Cherry Coke,0.134633,0.630075,soft drinks
4,Vanilla Coke Zero,0.130740,0.611854,soft drinks



Product: Plain Bagels  (ID: 20738)


,sub_name,transferability_pct,score,aisle
0,Plain Mini Bagels,0.154851,0.737898,breakfast bakery
1,Plain Pre-Sliced Bagels,0.154590,0.736655,breakfast bakery
2,Assorted Bagels,0.145627,0.693944,breakfast bakery
3,Bagels Plain Presliced,0.143973,0.686062,breakfast bakery
4,Onion Bagels,0.138857,0.661686,breakfast bakery



Product: Banana  (ID: 24852)


...

,sub_name,transferability_pct,score,aisle
0,Spaghetti Pasta,0.193824,0.789977,dry pasta
1,Thin Spaghetti Pasta,0.172642,0.703646,dry pasta
2,Whole Grain Spaghetti,0.161338,0.657572,dry pasta
3,Whole Wheat Spaghetti,0.155815,0.635062,dry pasta
4,Penne Rigate,0.106358,0.433486,dry pasta



Product: Sea Salt & Vinegar Potato Chips  (ID: 40709)


,sub_name,transferability_pct,score,aisle
0,Vinegar & Sea Salt Potato Chips,0.162931,0.753618,chips pretzels
1,Kettle Cooked Potato Chips Sea Salt and Vinegar,0.149928,0.693473,chips pretzels
2,Sea Salt Baked Potato Chips,0.148982,0.689101,chips pretzels
3,Salt & Pepper Krinkle Chips,0.146159,0.676044,chips pretzels
4,"Potato Chips, Lightly Salted",0.145618,0.673540,chips pretzels



Product: Diet Coke  (ID: 43631)


,sub_name,transferability_pct,score,aisle
0,Diet Cola,0.184405,0.680563,soft drinks
1,Diet Pepsi Soda,0.152335,0.562203,soft drinks
2,Soda,0.118957,0.439019,soft drinks
3,Fridge Pack Cola,0.112507,0.415215,soft drinks
4,Ginger Ale,0.112360,0.414673,soft drinks


Compute pairwise probabilities for the substitutes of the focus products.

In [13]:
%run ../utils/pairwise.py

all_subs = pd.read_csv("../results/substitutes-transfer.csv")
subs_list = all_subs['substitute_id'].unique().tolist()

order_products_prior = pd.read_csv('../dataset/order_products__prior.csv')
order_product_df = order_products_prior[['order_id', 'product_id']]

# Compute pairwise probabilties to be used for later calculations
product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    subs_list,
    output_csv="../results/subs-pairwise.csv",
    batch_size=1000
)

100%|██████████| 1/1 [00:07<00:00,  7.88s/it]

Completed computation. Saved to ../results/subs-pairwise.csv


Compute complement impact for focus products and substitutes.

In [15]:
%run ../utils/complements.py
%run ../utils/results.py

sample_df = pd.read_csv("../results/sampled-products.csv")
focus_pairwise_df = pd.read_csv("../results/products-pairwise.csv")
sub_pairwise_df = pd.read_csv("../results/subs-pairwise.csv")

pairwise_df = pd.concat([focus_pairwise_df, sub_pairwise_df], ignore_index=True)

num_orders, min_pij = get_min_pij()
focus_products = product_ids.copy()
all_products = subs_list + focus_products

lift_df = compute_lift(all_products, pairwise_df, min_pij=min_pij, total_orders=num_orders)
complements_df = compute_hybrid_score(lift_df, all_products, top_n=10)
network_df = compute_network_enhanced_impact(complements_df, pairwise_df)
network_df.to_csv("../results/complements-impact.csv", index=False)

#show_comp_results(network_df)

Compute total impact for focus products.

In [ ]:
product_info_df = pd.read_csv('../data/cleaned/product-features.csv')

totals = compute_total_impact(
    pairwise_impact_df=network_df,        
    penetration_df=product_info_df, 
    top_k=5
)
totals.to_csv("../results/complement-total-impact.csv", index=False)

Compute complement impact for 5000 product sample.

In [24]:
sample_df = pd.read_csv("../data/validation/sampled-products.csv")
pairwise_df = pd.read_csv("../data/validation/sample-pairwise.csv")

num_orders, min_pij = get_min_pij()
focus_products = sample_df['product_id'].unique().tolist()

lift_df = compute_lift(focus_products, pairwise_df, min_pij=min_pij, total_orders=num_orders)
complements_df = compute_hybrid_score(lift_df, focus_products, top_n=10)
network_df = compute_network_enhanced_impact(complements_df, pairwise_df)
network_df.to_csv("../results/sample-complements-impact.csv", index=False)

Compute total complement impact for 5000 product sample.

In [25]:
product_info_df = pd.read_csv('../data/cleaned/product-features.csv')

totals = compute_total_impact(
    pairwise_impact_df=network_df,        
    penetration_df=product_info_df, 
    top_k=5
)
totals.to_csv("../results/sample-total-impact.csv", index=False)


Compute statistics for substitution.

In [21]:
import matplotlib.pyplot as plt


# Build substitutes df
combined_subs_df = []

for i in range(1, 22):
    df = pd.read_csv(f"../data/validation/substitutes/dept{i}-transfer.csv")
    df['department_id'] = i
    combined_subs_df.append(df)

substitutes_df = pd.concat(combined_subs_df, ignore_index=True)

# Substitute score distribution
print("*** Substitution score distribution ***")
display(substitutes_df['score'].describe().to_frame(name='value'))

plt.hist(substitutes_df['score'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Products")
plt.ylabel("Substitution Score")
plt.title("Distribution of Substitution Score")
plt.savefig(f"../graphs/substitution_hist.png") 
plt.close()

# Transferability pct distribution
print("*** Transferability pct distribution ***")
display(substitutes_df['transferability_pct'].describe().to_frame(name='value'))

plt.hist(substitutes_df['transferability_pct'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Products")
plt.ylabel("Transfer %")
plt.title("Distribution of Transfer %")
plt.savefig(f"../graphs/transfer_hist.png") 
plt.close()

# Compute per-product metrics
product_metrics = substitutes_df.groupby(['department_id', 'product_id']).agg(
    avg_score=('score', 'mean'),
    avg_transferability_pct=('transferability_pct', 'mean'),
    total_transferability_pct=('transferability_pct', 'sum'),
    num_subs=('substitute_id', 'nunique')
).reset_index()

# Total transferability distribution
print("*** Total Transferability distribution ***")
display(product_metrics['total_transferability_pct'].describe().to_frame(name='value'))

plt.hist(product_metrics['total_transferability_pct'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Products")
plt.ylabel("Total Transfer %")
plt.title("Distribution of Total Transfer %")
plt.savefig(f"../graphs/total_transfer_hist.png") 
plt.close()

# Aggregate to department level
department_metrics = product_metrics.groupby('department_id').agg(
    avg_score=('avg_score', 'mean'),
    avg_transferability_pct=('avg_transferability_pct', 'mean'),
    avg_num_subs=('num_subs', 'mean'),
    avg_total_transferability_pct=('total_transferability_pct', 'mean'),
    num_sample_products=('product_id', 'nunique')
).reset_index()

department_metrics = department_metrics.merge(departments, on='department_id', how='left')

#department_metrics.to_csv("../results/department-subs.csv")

display(department_metrics)

*** Substitution score distribution ***


,value
count,17469.000000
mean,0.485436
std,0.108958
min,0.400000
25%,0.418940
50%,0.434314
75%,0.448278
max,0.831380


*** Transferability pct distribution ***


,value
count,17469.000000
mean,0.136788
std,0.077697
min,0.081587
25%,0.088184
50%,0.114420
75%,0.148660
max,0.784520


*** Total Transferability distribution ***


,value
count,4504.000000
mean,0.530539
std,0.123788
min,0.400000
25%,0.440000
50%,0.440000
75%,0.667221
max,0.831380


,department_id,avg_score,avg_transferability_pct,avg_num_subs,avg_total_transferability_pct,num_sample_products,department
0,1,0.470167,0.153029,4.257979,0.527181,376,frozen
1,2,0.438766,0.360418,1.583333,0.450748,24,other
2,3,0.460915,0.194492,3.720588,0.511424,136,bakery
3,4,0.466239,0.119221,4.739130,0.526498,161,produce
4,5,0.470847,0.184694,3.773333,0.536600,75,alcohol
5,6,0.448745,0.183023,3.764706,0.496956,102,international
6,7,0.484425,0.155040,4.272289,0.546002,415,beverages
7,8,0.487184,0.169861,3.980392,0.531894,102,pets
8,9,0.486860,0.166524,4.140127,0.553526,157,dry goods pasta
9,10,0.413894,0.265889,2.500000,0.421371,4,bulk


Calculate statistics for Total CII.

In [28]:
products_details = (
    product_df[['product_id','product_name','aisle_id','department_id']]
      .merge(aisles, on='aisle_id', how='left')
      .merge(departments, on='department_id', how='left')
)
products_details.drop(['aisle_id','department_id'], axis=1, inplace=True)
comp_total_df = pd.read_csv(f"../results/sample-total-impact.csv")
comp_total_df = comp_total_df.merge(products_details, on="product_id", how="left")

# Distribution of total complement impact
print("*** Total Complement Impact Index distribution ***")
display(comp_total_df['total_impact_norm'].describe())

plt.hist(comp_total_df['total_impact_norm'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Products")
plt.ylabel("Total CII")
plt.title("Distribution of Total CII")
plt.savefig(f"../graphs/total_cii_hist.png") 
plt.close()

dept_total_comp = comp_total_df.groupby('department').agg(
    avg_total_impact=('total_impact_norm', 'mean'),
    num_sample_products=('product_id', 'nunique')
).reset_index()

print("*** Avg Total Complement Impact Index by Dept *** ")
print(dept_total_comp[["department", "avg_total_impact"]])

*** Total Complement Impact Index distribution ***


KeyError: 'total_impact_norm'

Calculate statistics for CII.

In [ ]:
import pandas as pd

comp_df = pd.read_csv(f"../data/validation/complements/sample-complements.csv")
comp_df = comp_df[['product_id','complement_id', 'impact_pct_j','impact_index_j']]

# Distribution of Complement Impact Index
print("*** Complement Impact Index distribution ***")
display(comp_df['impact_index'].describe())

plt.hist(comp_df['impact_index'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Products")
plt.ylabel("CII")
plt.title("Distribution of CII")
plt.savefig(f"../graphs/cii_hist.png") 
plt.close()

product_depts = (
    product_df[['product_id','product_name','department_id']]
      .merge(departments, on='department_id', how='left')
)
df = comp_df.merge(product_depts.rename(columns={'product_id': 'product_id', 'department': 'prod_dept'}),
                          on='product_id', how='left')

print(df.columns)
df = df.merge(product_depts.rename(columns={'product_id': 'complement_id', 'department': 'comp_dept'}),
              on='complement_id', how='left')


# Count occurrences of comp_dept per prod_dept
counts = df.groupby(['prod_dept', 'comp_dept']).size().reset_index(name='count')

# Get total complements per prod_dept
total_per_prod = counts.groupby('prod_dept')['count'].transform('sum')

# Compute percentage
counts['percentage'] = counts['count'] / total_per_prod * 100

# Sort for readability
counts = counts.sort_values(['prod_dept', 'percentage'], ascending=[True, False])

print(counts.head())

for prod, group in counts.groupby('prod_dept'):
    print(f"{prod}")  # header
    for row in group.itertuples():
        print(f"  - {row.comp_dept}: {row.percentage:.1f}%")
    print()  # blank line between departments





*** Complement Impact Index distribution ***
count    10830.000000
mean         0.019311
std          0.047916
min          0.000028
25%          0.002610
50%          0.006996
75%          0.017342
max          1.000000
Name: impact_index_j, dtype: float64
Index(['product_id', 'complement_id', 'impact_pct_j', 'impact_index_j',
       'product_name', 'department_id', 'prod_dept'],
      dtype='object')
  prod_dept        comp_dept  count  percentage
0   alcohol          alcohol     56   80.000000
2   alcohol       dairy eggs      6    8.571429
3   alcohol  dry goods pasta      2    2.857143
7   alcohol           snacks      2    2.857143
1   alcohol           bakery      1    1.428571
alcohol
  - alcohol: 80.0%
  - dairy eggs: 8.6%
  - dry goods pasta: 2.9%
  - snacks: 2.9%
  - bakery: 1.4%
  - other: 1.4%
  - pets: 1.4%
  - produce: 1.4%

babies
  - babies: 77.1%
  - snacks: 6.2%
  - frozen: 2.4%
  - missing: 2.4%
  - dairy eggs: 1.9%
  - bakery: 1.4%
  - beverages: 1.4%
  - breakfast